# QC Benchmark — Hybrid Recordings

Trains and evaluates on **hybrid recordings only** (HYBRID_* study sets).
Paired and synth data are excluded.

Hybrid recordings inject synthetic spikes into real extracellular noise at known locations.
Ground truth is controlled by construction — this provides the largest training set
and the cleanest label signal.

In [ ]:
import importlib, subprocess, sys
for pkg in ["lightgbm", "shap", "seaborn", "pyarrow", "scipy"]:
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])


In [ ]:
import sys, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from qc_framework import QCBenchmarkPipeline
from qc_framework.evaluation import SHAPAnalyzer
from qc_framework.pipeline import QCDatasetReviewPipeline

warnings.filterwarnings("ignore", message="X does not have valid feature names")
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 130

# ═══════════════════════════════════════════════════════
# CONFIG
# ═══════════════════════════════════════════════════════
TRAIN_PARQUET_PATH   = PROJECT_ROOT / "data" / "7900_ROWS.parquet"
TEST_PARQUET_PATH    = None           # path to external test parquet, or None

DATASET_TYPES        = ["hybrid"]     # edit to add/remove types
TARGETS              = ["accuracy", "fpos", "fmiss_extended"]  # targets to model
WEIGHTING_STRATEGY   = None           # None | "paired_3x" | "inverse_type" | "inverse_recording"
EXCLUDE_SOFT_CONTEXT = False          # True drops rec_n_channels, rec_sampling_rate, rec_duration_sec
RANDOM_STATE         = 42
GROUPED_TEST_SIZE    = 0.20
HOLDOUT_SORTER_MIN   = 170            # minimum rows per sorter to run sorter holdout

OUTPUT_DIR           = None           # None = auto: analysis/runs/<train_stem>_hybrid/outputs
# ═══════════════════════════════════════════════════════

assert TRAIN_PARQUET_PATH.exists(), f"Missing: {TRAIN_PARQUET_PATH}"
if TEST_PARQUET_PATH is not None:
    assert Path(TEST_PARQUET_PATH).exists(), f"Missing: {TEST_PARQUET_PATH}"
if OUTPUT_DIR is None:
    OUTPUT_DIR = PROJECT_ROOT / "analysis" / "runs" / (TRAIN_PARQUET_PATH.stem.lower() + "_hybrid") / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Train :", TRAIN_PARQUET_PATH)
print("Test  :", TEST_PARQUET_PATH)
print("Types :", DATASET_TYPES)
print("Output:", OUTPUT_DIR)


## 1. Load and Clean

In [ ]:
pipeline = QCBenchmarkPipeline(
    train_path=TRAIN_PARQUET_PATH,
    test_path=TEST_PARQUET_PATH,
    random_state=RANDOM_STATE,
    grouped_test_size=GROUPED_TEST_SIZE,
    weighting_strategy=WEIGHTING_STRATEGY,
    exclude_soft_context=EXCLUDE_SOFT_CONTEXT,
    holdout_sorter_min_rows=HOLDOUT_SORTER_MIN,
    dataset_type_filter=DATASET_TYPES,
)

train_clean, test_clean, clean_report = pipeline.load_and_clean()

print(f"Rows: {len(train_clean):,}  |  Recordings: {train_clean['recording_key'].nunique()}  |  Sorters: {train_clean['sorter_name'].nunique()}")
if "study_set" in train_clean.columns:
    display(train_clean["study_set"].value_counts().rename("rows").to_frame())
display(clean_report["drop_summary"])

available_targets = [t for t in TARGETS if t in train_clean.columns]
display(QCDatasetReviewPipeline(TRAIN_PARQUET_PATH).target_summary(train_clean, available_targets))


## 2. Feature Spaces

In [ ]:
specs = pipeline.build_feature_specs(train_clean)
display(pd.DataFrame([
    {"mode": s.mode, "exclude_soft": bool(s.soft_excluded_cols),
     "n_numeric": len(s.numeric_cols), "n_categorical": len(s.categorical_cols),
     "n_total": len(s.all_features)}
    for s in specs
]))


## 3. Split

In [ ]:
grouped_train, grouped_test, row_random_train, row_random_test, split_report = pipeline.split(train_clean)

print(f"Grouped  — train: {len(grouped_train):,} rows / {grouped_train['recording_key'].nunique()} recordings")
print(f"          test:  {len(grouped_test):,} rows / {grouped_test['recording_key'].nunique()} recordings")

available_targets = [t for t in TARGETS if t in grouped_train.columns]
review = QCDatasetReviewPipeline(TRAIN_PARQUET_PATH)
print("\nTrain target summary:")
display(review.target_summary(grouped_train, available_targets))
print("Test target summary:")
display(review.target_summary(grouped_test, available_targets))


## 4. Baseline Models

In [ ]:
results = pipeline.fit_baseline(
    grouped_train, grouped_test, row_random_train, row_random_test, specs,
    targets=TARGETS,
)
results_df = pipeline.results_table(results)
display(results_df)


In [ ]:
available_targets = [t for t in TARGETS if t in grouped_train.columns]
fig, axes = plt.subplots(1, len(available_targets), figsize=(5 * len(available_targets), 4.5))
if len(available_targets) == 1:
    axes = [axes]
for ax, target in zip(axes, available_targets):
    result = next((r for r in results if r.target == target
                   and r.split_name == "grouped_recording"
                   and r.feature_mode == "unit_only"
                   and r.model_name == "lightgbm"), None)
    if result is None:
        ax.axis("off")
        continue
    p = result.predictions
    ax.scatter(p["true"], p["pred"], s=16, alpha=0.3, color="#1d3557")
    ax.plot([0, 1], [0, 1], color="crimson", lw=2, ls="--")
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.set_title(f"{target}  MAE={result.mae:.3f}  R\u00b2={result.r2:.3f}")
    ax.set_xlabel("true"); ax.set_ylabel("pred")
fig.suptitle("Hybrid — LightGBM predictions (grouped/unit_only)", fontweight="bold")
plt.tight_layout()
plt.show()


## 5. Joint Target Distance

In [ ]:
from qc_framework.evaluation import JointEvaluator
joint = JointEvaluator().score(results, required_targets=tuple(t for t in TARGETS if t in grouped_train.columns))
display(joint)


## 6. Dataset-Type Transfer

Not applicable for single-type training — skipped.
Use `qc_modeling_benchmark.ipynb` for cross-type transfer experiments.

## 7. Sorter Transfer

In [ ]:
sorter_transfer = pipeline.sorter_transfer(train_clean, specs, targets=TARGETS)
display(sorter_transfer)

if not sorter_transfer.empty:
    st_plot = sorter_transfer.copy()
    st_plot["held_out_sorter"] = st_plot["split"].str.split(":").str[-1]
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    for ax, metric in zip(axes, ["mae", "rmse", "r2"]):
        sns.barplot(data=st_plot, x="held_out_sorter", y=metric, hue="target", ax=ax)
        ax.tick_params(axis="x", rotation=35)
        ax.set_title(f"Sorter Transfer: {metric.upper()}")
    plt.tight_layout()
    plt.show()


## 8. SHAP Feature Analysis

In [ ]:
shap_results = pipeline.shap_analysis(results, grouped_train, grouped_test, specs, targets=TARGETS)

for target, shap_df in shap_results.items():
    print(f"\nTop SHAP features — {target}")
    display(shap_df.head(20))
    family_df = SHAPAnalyzer.family_summary(shap_df)
    display(family_df)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    shap_df.head(20).set_index("feature")["mean_abs_shap"].sort_values().plot(kind="barh", ax=axes[0], color="#577590")
    axes[0].set_title(f"Top |SHAP|: {target}")
    family_df.set_index("family")["mean_abs_shap"].sort_values().plot(kind="barh", ax=axes[1], color="#84a98c")
    axes[1].set_title(f"SHAP by Family: {target}")
    plt.tight_layout()
    plt.show()


## 9. Save Artifacts

In [ ]:
import json as _json

for name, df in [
    ("baseline_results", results_df),
    ("joint_target_distance", joint),
    ("sorter_transfer", sorter_transfer),
]:
    if df is not None and not df.empty:
        df.to_csv(OUTPUT_DIR / f"{name}.csv", index=False)

if shap_results:
    payload = {t: df.head(25).set_index("feature")["mean_abs_shap"].to_dict()
               for t, df in shap_results.items() if not df.empty}
    (OUTPUT_DIR / "shap_top_features.json").write_text(_json.dumps(payload, indent=2))

print("Saved to:", OUTPUT_DIR)
for f in sorted(OUTPUT_DIR.iterdir()):
    print(" -", f.name)
